# abstractive summarization

In [ ]:
from transformers import pipeline
import os

# summarizer = pipeline("summarization")

## To use the t5-base model for summarization:
summarizer_2 = pipeline("summarization", model="t5-base", 
                      tokenizer="t5-base", framework="tf")


summary_page = []
page = -1
for pages in pagevector:
        page += 1 
        clean_text = summarizer_2(pagevector[page],
                               max_length = 100,
                                min_length=5,
                               do_sample =False)[0]['summary_text']
        summary_page.append(clean_text)
        
summary_page

In [1]:
row = 5
col = 5
for x in range(1, row + 1):
    for y in range(1, col + 1):
        print(" Row", x, end="")
        print(" Column", y, end="")
    print()

 Row 1 Column 1 Row 1 Column 2 Row 1 Column 3 Row 1 Column 4 Row 1 Column 5
 Row 2 Column 1 Row 2 Column 2 Row 2 Column 3 Row 2 Column 4 Row 2 Column 5
 Row 3 Column 1 Row 3 Column 2 Row 3 Column 3 Row 3 Column 4 Row 3 Column 5
 Row 4 Column 1 Row 4 Column 2 Row 4 Column 3 Row 4 Column 4 Row 4 Column 5
 Row 5 Column 1 Row 5 Column 2 Row 5 Column 3 Row 5 Column 4 Row 5 Column 5


In [5]:
import pandas as pd
import numpy as np
x = list('abc')
y = list('123')
data=[]
for i in x:
    row=[]
    for j in y:
        row.append(np.random.rand())
    data.append(row)
df = pd.DataFrame(data)#, index=x, columns=y)

In [6]:
df

,0,1,2
0,0.321548,0.328349,0.964548
1,0.243251,0.150115,0.057409
2,0.265258,0.569189,0.564647


In [8]:
range(0,(2294-1))

range(0, 2293)

In [ ]:
#!/usr/bin/env python
# coding: utf-8

# # NLP HMH 5th Grade ELA
# 
# 
# # Process
# 
# ## Intro: Data Ingestion and prep
# 
# - Install/Import necessary modules
# - Ingest CSV spine skills and descriptions
# - Ingest existent skill to content mapping
# - Translate existent mapping (IntoReading_G5_ELA-spine_Validation_Complete_TG.csv) into ground truth Y value we can optimize against.  Additionally correlation strength is a variable to consider for our purposes.
# - Ingest HMH 5th Grade ELA teacher guide pdf content as corpus for model training
# - Turn corpus into page vector
# 
# ## Encoding model
# - Count vectorizer
# - TFIDF
# - Word to vec (https://ai.intelligentonlinetools.com/ml/k-means-clustering-example-word2vec/)
# - BERT
# - BART
# 
# ## Dimension Reduction
# https://elitedatascience.com/dimensionality-reduction-algorithms
# - PCA
# - LDA
# - Truncated SVD
# - UMAP
# - t-SNE
# 
# ## Clustering Algorithm
# - K Means
# - Kcluster (https://pypi.org/project/kcluster/)
# - DBSCAN
# - HDBSCAN
# 
# ## Baseline model
# - Word vectorizer = count vectorizer
# - Dimension reduction = PCA
# - Clustering = k-means
# - Create baseline topic model 
# 
# ## Model 2
# 
# ## Model 3

# ## Intro: Data Ingestion and prep

# #### Install/Import necessary modules

# In[1]:


#!pip install PyPDf2
#!pip install transformers
#!pip install torch


# In[2]:


#import
import numpy as np
import pandas as pd
import json
import os
import PyPDF2
#from top2vec import Top2Vec
from transformers import BertTokenizer, BertModel
import seaborn as sns
import matplotlib.pyplot as plt
import re 
from pandas import DataFrame
import torch
import sklearn
from transformers import Trainer, TrainingArguments
get_ipython().system('pip install pandas-profiling')
import pandas_profiling


# #### Ingest CSV spine skills and descriptions

# In[6]:


spine = pd.read_excel(r'C:\Users\rdominguez\Documents\Pers\UChicago\Capstone\Data\OneCMS_Learning_Spine_English_Language_Arts2021012117.xls', skiprows=5)
trimmed_spine_columns = ['Skill Title', 'Skill Code', 'Skill Description']
spine_trimmed = spine[trimmed_spine_columns]
#print(spine_trimmed.head())


# In[7]:


spine_trimmed.profile_report()


# #### Ingest existent skill to content mapping

# In[4]:


spine_validation = pd.read_excel(r'C:\Users\rdominguez\Documents\Pers\UChicago\Capstone\Data\IntoReading_G5_ELA-spine_Validation_Complete_TG.xls')
trimmed_spine_validation_columns = ['Skill GUID', ' Skill Title', ' Skill Code', ' Skill Description', 'Correlation Strength', 'Add/ Delete/ Change', 'Mod.', 'Wk.']
spine_validation_trimmed = spine_validation[trimmed_spine_validation_columns]

print(spine_validation_trimmed.describe())
print()
print(spine_validation_trimmed.nunique()) #why mismatch between title, code, and descriptions?
print()
print(spine_validation_trimmed['Wk.'].unique())
print()
print(spine_validation_trimmed['Mod.'].unique())
print()
print(spine_validation_trimmed[' Skill Code'].value_counts())

fig, distplot = plt.subplots(figsize = (14,7))
distplot = sns.distplot(spine_validation_trimmed[' Skill Code'].value_counts())


# #### Translate existent mapping (IntoReading_G5_ELA-spine_Validation_Complete_TG.csv) into ground truth Y value we can optimize against.  Additionally correlation strength is a variable to consider for our purposes.

# In[ ]:





# #### Ingest HMH 5th Grade ELA teacher guide pdf content as corpus for model training

# In[5]:


pdfFileObj = open('HMH_IntoReading_G5_TeachersGuide_v01.pdf','rb')
pdfReader = PyPDF2.PdfFileReader(pdfFileObj)

pagecount = pdfReader.getNumPages ()

pagevector = []
for count in range(pagecount):
    page = pdfReader.getPage(count)
    page_extract = page.extractText ()
    pagevector.append(page_extract)
#pagevector


# In[6]:


import operator

module_start_pages = []
page_iterator = 0
text = "Module [0-9]+ Ł\n Lesson [0-9]+"
resources_text = "Resources"

def new_week_logic(module_lesson_input):
    week1 = 'Lesson ?[1-5]{1}'
    week2 = 'Lesson ?[6-9]{1}|Lesson 10'
    week3 = 'Lesson 1[1-5]{1}'
    intro = 'Intro'
    resources = 'Resources'
    match_week1 = re.search(week1, module_lesson_input)
    match_week2 = re.search(week2, module_lesson_input)
    match_week3 = re.search(week3, module_lesson_input)
    match_intro = re.search(intro, module_lesson_input)
    match_resources = re.search(resources,module_lesson_input)
    if match_week2 != None: week = "02"
    elif match_week3 != None: week = "03"
    elif match_week1 != None: week = "01"
    elif match_intro != None: week = "Intro"
    elif match_resources != None: week = "Resources"
    else: week = "NULL"
    return week

for page in pagevector:
    page_iterator += 1
    if 'text' in page.partition('\n')[-1]:
            match = re.search(text, page.partition('\n')[-1])
            if match != None:
                module_lesson = str(re.findall(text,page.partition('\n')[-1],flags = re.I + re.M))
            else: 
                module_lesson = str(module_start_pages[-1][1])
            week = new_week_logic(module_lesson)
            module_week_mapping = [page_iterator, module_lesson,str(week),page]
            module_start_pages.append(module_week_mapping)
    else:
        if len(module_start_pages) == 0:
              module_lesson = 'Intro'
              week = 'Intro'
              module_start_pages.append([page_iterator,module_lesson,week,page])
        if len(module_start_pages) > 0:
              module_lesson = module_start_pages[-1][1]
              week = module_start_pages[-1][2]
              module_start_pages.append([page_iterator,module_lesson,week,page])
        else:
              module_lesson = 'NULL'
              week = week_logic
              module_start_pages.append([page_iterator,module_lesson,str(week),page])
            
module_start_pages


# In[7]:


#convert list to DataFrame
#df = DataFrame(module_start_pages)
#len(df[df[2]=='Intro'])/len(df)


# In[8]:


#pagevector[420]


# In[9]:


df.head(100)


# Split data into train, val, and test

# ## Baseline BERT model

# In[ ]:


import torch
from tqdm.notebook import tqdm

from transformers import BertTokenizer
from torch.utils.data import TensorDataset

from transformers import BertForSequenceClassification


# In[ ]:


sentences= df[3].values
labels = df[1].values
#df.head()

from transformers import BertTokenizer
# using the low level BERT for our task.
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)


# In[ ]:


split = np.random.choice(
    ["train", "val", "test"],
    size=df.shape[0],
    p=[.7, .15, .15])
df["split"] = split
x_train = df[df["split"] == "train"]
x_test = df[df["split"] == "test"]
y_train = x_train[1]
y_test = x_test[1]


# In[ ]:


encoded_data_train = tokenizer.batch_encode_plus(
    x_train[3].values, 
    add_special_tokens=True, 
    return_attention_mask=True, 
    pad_to_max_length=True, 
    max_length=512, 
    return_tensors='pt',
    truncation=True)

encoded_data_val = tokenizer.batch_encode_plus(
    x_test[3].values, 
    add_special_tokens=True, 
    return_attention_mask=True, 
    pad_to_max_length=True, 
    max_length=512, 
    return_tensors='pt',
    truncation=True)

input_ids_train = encoded_data_train['input_ids']
attention_masks_train = encoded_data_train['attention_mask']
labels_train = pd.factorize(y_train)[0]
labels_train = torch.tensor(labels_train)

input_ids_val = encoded_data_val['input_ids']
attention_masks_val = encoded_data_val['attention_mask']
labels_val = pd.factorize(y_test)[0]
labels_val = torch.tensor(labels_val)

dataset_train = TensorDataset(input_ids_train, attention_masks_train, labels_train)
dataset_val = TensorDataset(input_ids_val, attention_masks_val, labels_val)


# class NewsGroupsDataset(torch.utils.data.Dataset):
#     def __init__(self, encodings, labels):
#         self.encodings = encodings
#         self.labels = labels
# 
#     def __getitem__(self, idx):
#         item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
#         item["labels"] = torch.tensor([self.labels[idx]])
#         return item
# 
#     def __len__(self):
#         return len(self.labels)
# 
# # convert our tokenized data into a torch Dataset
# train_dataset = NewsGroupsDataset(encoded_data_train, labels_train)
# valid_dataset = NewsGroupsDataset(encoded_data_val, labels_val)

# In[ ]:


len(set(labels_train))
len(set(df[1]))


# In[ ]:


len(labels_train)


# In[ ]:


model_name = "bert-base-uncased"
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=1)


# In[ ]:


from sklearn.metrics import accuracy_score

def compute_metrics(pred):
  labels = pred.label_ids
  preds = pred.predictions.argmax(-1)
  # calculate accuracy using sklearn's function
  acc = accuracy_score(labels, preds)
  return {
      'accuracy': acc,
  }


# In[ ]:


training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=3,              # total number of training epochs
    per_device_train_batch_size=16,  # batch size per device during training
    per_device_eval_batch_size=20,   # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    load_best_model_at_end=True,     # load the best model when finished training (default metric is loss)
    # but you can specify `metric_for_best_model` argument to change to accuracy or other metric
    logging_steps=200,               # log & save weights each logging_steps
    evaluation_strategy="steps",     # evaluate each `logging_steps`
)


# In[ ]:


trainer = Trainer(
    model=model,                         # the instantiated Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=train_dataset,         # training dataset
    eval_dataset=valid_dataset,          # evaluation dataset
    compute_metrics=compute_metrics,     # the callback that computes metrics of interest
)


# In[ ]:


for k, v in train_dataset.encodings.items():
    print(k,v)
    
    
for k, v in valid_dataset.encodings.items():
    print(k,v)


# In[ ]:


print(train_dataset.encodings)


# In[ ]:


trainer.train()
#Trainer(model=model,train_dataset=dataset_train,eval_dataset=dataset_val).train()


# In[ ]:


for data in train_dataset:
    print(len(data[1]))
for data in valid_dataset:
    print(data)


# In[ ]:


small_pagevector = pagevector[197:200]


# In[ ]:


tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

tokenized_dict = tokenizer.encode_plus(pagevector,add_special_tokens=True,max_length=30,Truncation=True)

bert_model = BertModel.from_pretrained('bert-base-uncased')
#inputs = tokenizer(small_pagevector, return_tensors="pt",truncation=True, padding=True)
#outputs = model(**inputs)

tokenized_text = torch.tensor(tokenized_dict["input_ids"])
#with torch.no_grad():
#  embeddings = bert_model(torch.tensor(tokenized_text.unsqueeze(0)))


# # Top2Vec

# model = Top2Vec(pagevector,hierarchical_topic_reduction=True)

# model.get_num_topics()

# topic_sizes, topic_nums = model.get_topic_sizes()

# topic_sizes

# topic_nums

# model

# print(model)

# dir(Top2Vec)

# dir(model)

# model.get_topic_hierarchy()

# model.get_topics()

# list_of_strings = ["Identify","Purpose"]
# model.search_documents_by_keywords(list_of_strings, pagecount)

# model.topic_words

# num_sim = 10
# model.similar_words(list_of_strings, num_sim)

# model.embedding_model

# #model.doc_dist
# model.doc_id2index

#  'doc_dist_reduced',
#  'doc_id2index_id',
#  'doc_id_type',

# Once we have a methodology for testing the model we can proceed to delete pages we consider irrelevant for model training via model.delete_documents(doc_ids)

# model.delete_documents()

# Code below for searching for keywords
# 
# 
# documents, document_scores, document_ids = model.search_documents_by_keywords(keywords=["cryptography", "privacy"], num_docs=5)
# for doc, score, doc_id in zip(documents, document_scores, document_ids):
#     print(f"Document: {doc_id}, Score: {score}")
#     print("-----------")
#     print(doc)
#     print("-----------")
#     print()


In [12]:
import autoimpute

In [13]:
import xgboost

XGBoostError: XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed (vcomp140.dll or libgomp-1.dll for Windows, libomp.dylib for Mac OSX, libgomp.so for Linux and other UNIX-like OSes). Mac OSX users: Run `brew install libomp` to install OpenMP runtime.
  * You are running 32-bit Python on a 64-bit OS
Error message(s): ['dlopen(/Library/Frameworks/Python.framework/Versions/3.8/lib/python3.8/site-packages/xgboost/lib/libxgboost.dylib, 6): Library not loaded: /usr/local/opt/libomp/lib/libomp.dylib\n  Referenced from: /Library/Frameworks/Python.framework/Versions/3.8/lib/python3.8/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: image not found']
